# Hafta 4 — Özellik Mühendisliği ve Ridge Regression Baseline

Bu notebook, önceden temizlenmiş Ames Housing verisini (`df_cleaned`) kullanır. Önceki veri temizleme hücrelerini bu notebook'tan önce çalıştırın veya `df_cleaned` değişkenini oluşturun.

In [ ]:
# Hafta 4 — Özellik Mühendisliği ve Ridge Regression Baseline
# Gerekli giriş: df_cleaned adlı, temizlenmiş Ames Housing DataFrame'i.
# df_cleaned; SalePrice hedef sütununu ve aşağıda kullanılan sütunları içermelidir.

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


class SkewedLog1pTransformer(BaseEstimator, TransformerMixin):
    """Fit edildiği eğitim verisinde çarpık ve negatif olmayan sütunları seçip log1p uygular."""

    def __init__(self, skew_threshold=0.75, min_unique=3):
        self.skew_threshold = skew_threshold
        self.min_unique = min_unique

    def fit(self, X, y=None):
        X_array = np.asarray(X, dtype=float)
        self.n_features_in_ = X_array.shape[1]
        self.skewed_indices_ = []

        for i in range(X_array.shape[1]):
            col = X_array[:, i]
            valid = col[np.isfinite(col)]
            if len(valid) == 0 or np.unique(valid).size < self.min_unique:
                continue
            # log1p negatif değerlerde geçerli olmadığından yalnızca min >= 0 ise seç.
            if valid.min() < 0:
                continue
            skew_value = pd.Series(valid).skew()
            if np.isfinite(skew_value) and abs(skew_value) > self.skew_threshold:
                self.skewed_indices_.append(i)
        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=float).copy()
        if X_array.shape[1] != self.n_features_in_:
            raise ValueError("Girdi sütun sayısı fit sırasında görülen sayı ile eşleşmiyor.")
        for i in self.skewed_indices_:
            # İmputer önce çalıştığı için normalde NaN kalmaz; güvenlik için kontrol edilir.
            if np.any(X_array[:, i] < 0):
                raise ValueError("log1p uygulanacak sütunda negatif değer bulundu.")
            X_array[:, i] = np.log1p(X_array[:, i])
        return X_array


# -------------------------------------------------------------------
# 1. Veri kontrolü ve kopyalama
# -------------------------------------------------------------------
if "df_cleaned" not in globals():
    raise NameError(
        "df_cleaned bulunamadı. Önce veri temizleme adımlarını çalıştırıp "
        "temizlenmiş DataFrame'i df_cleaned değişkenine atayın."
    )

df = df_cleaned.copy()

required_columns = [
    "SalePrice", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath",
    "YrSold", "YearBuilt", "YearRemodAdd", "OpenPorchSF",
    "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
    "GarageArea", "Fireplaces", "Id", "TotRmsAbvGrd",
    "GarageYrBlt", "Neighborhood", "LotFrontage",
    "MSSubClass", "ExterQual", "ExterCond", "BsmtQual",
    "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu",
    "GarageQual", "GarageCond", "MasVnrArea"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(
        "Veri setinde gerekli sütunlar eksik: " + ", ".join(missing_columns)
    )

if df["SalePrice"].isna().any():
    raise ValueError("SalePrice sütununda eksik değer var; hedef değişken temizlenmeli.")
if (df["SalePrice"] < 0).any():
    raise ValueError("log1p dönüşümü için SalePrice negatif olamaz.")


# -------------------------------------------------------------------
# 2. Yeni özellikler
# -------------------------------------------------------------------
# Eksik alan değerleri bu toplam özelliklerde 0 kabul edilir; veri setinde
# bu alanların eksikliği gerçekten 'alan yok' anlamına geliyorsa uygundur.
df["TotalSF"] = (
    df["TotalBsmtSF"].fillna(0)
    + df["1stFlrSF"].fillna(0)
    + df["2ndFlrSF"].fillna(0)
)
df["TotalBath"] = (
    df["FullBath"].fillna(0)
    + 0.5 * df["HalfBath"].fillna(0)
    + df["BsmtFullBath"].fillna(0)
    + 0.5 * df["BsmtHalfBath"].fillna(0)
)
df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)
df["RemodAge"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)
porch_columns = ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
df["TotalPorchSF"] = df[porch_columns].fillna(0).sum(axis=1)
df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype(int)
df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)

print("TotalSF - SalePrice korelasyonu:", round(df["TotalSF"].corr(df["SalePrice"]), 3))

# Hedef değişkenin log dönüşümü
y = np.log1p(df["SalePrice"])
print(
    "SalePrice çarpıklığı:",
    round(df["SalePrice"].skew(), 3),
    "-> log1p sonrası:",
    round(y.skew(), 3)
)


# -------------------------------------------------------------------
# 3. Girdi sütunlarını hazırlama
# -------------------------------------------------------------------
drop_columns = [
    "SalePrice", "Id", "GarageArea", "TotRmsAbvGrd",
    "1stFlrSF", "TotalBsmtSF", "GarageYrBlt"
]
X = df.drop(columns=drop_columns).copy()

# Ames Housing veri setinde sayısal kodlanmış sınıf bilgisi kategorik ele alınır.
X["MSSubClass"] = X["MSSubClass"].astype("string")

quality_columns = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
    "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond"
]
quality_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
for col in quality_columns:
    X[col] = X[col].fillna("None").map(quality_map)

# Dış test kümesi ayrılır; öğrenilen dönüşümler daha sonra yalnızca train'de fit edilir.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Eğitim kayıt sayısı: {len(X_train)} | Test kayıt sayısı: {len(X_test)}")


# -------------------------------------------------------------------
# 4. Ön işleme ve model Pipeline'ı
# -------------------------------------------------------------------
numeric_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p_skewed", SkewedLog1pTransformer(skew_threshold=0.75, min_unique=3)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=10))
])

model.fit(X_train, y_train)

feature_count = len(model.named_steps["preprocessor"].get_feature_names_out())
print("Kodlama sonrası özellik sayısı:", feature_count)

# Hangi sayısal sütunlara log1p uygulandığını göster (final modelin train-fit sonucu)
log_indices = model.named_steps["preprocessor"].named_transformers_["num"] \
    .named_steps["log1p_skewed"].skewed_indices_
print("log1p uygulanan sayısal sütunlar:", [numeric_columns[i] for i in log_indices])


# -------------------------------------------------------------------
# 5. Test kümesi değerlendirmesi
# -------------------------------------------------------------------
pred_log = model.predict(X_test)
# Log hedefin ters dönüşümü: tahminleri dolar ölçeğine getir.
pred_price = np.maximum(0, np.expm1(pred_log))
true_price = np.expm1(y_test)

mae = mean_absolute_error(true_price, pred_price)
rmse = np.sqrt(mean_squared_error(true_price, pred_price))
r2 = r2_score(true_price, pred_price)

print(f"Test MAE : ${mae:,.0f}")
print(f"Test RMSE: ${rmse:,.0f}")
print(f"Test R²  : {r2:.3f}")


# -------------------------------------------------------------------
# 6. 5 katlı çapraz doğrulama
# -------------------------------------------------------------------
# Pipeline içindeki imputation, log-sütun seçimi, ölçekleme ve encoding
# her CV katının eğitim parçasında ayrı ayrı fit edilir.
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse_scores = -cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv_splitter,
    scoring="neg_root_mean_squared_error"
)
print(f"5 katlı CV RMSE (log ölçeği): {cv_rmse_scores.mean():.3f}")
print(f"CV RMSE standart sapması: {cv_rmse_scores.std():.3f}")
